In [1]:
!pip install -q sentence-transformers

In [4]:
import time, platform
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score

SEED = 42
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
FILES = {"Small": "starwars_clean_small_125.csv", "Large": "starwars_clean_large_4char.csv"}


def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0], zero_division=0)
    return {"Accuracy": accuracy_score(y_true, y_pred), "Bad Precision": p[0], "Bad Recall": r[0],
            "Bad F1": f[0], "Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0)}


def make_clf(name, balanced):
    cw = "balanced" if balanced else None
    if name == "LogReg":
        return LogisticRegression(max_iter=2000, class_weight=cw, random_state=SEED)
    return LinearSVC(class_weight=cw, random_state=SEED)


def cv_eval(make_estimator, X, y):
    """X는 문장 배열(TF-IDF 파이프라인) 또는 임베딩 행렬(SBERT). 폴드별 지표 DataFrame 반환."""
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
    rows = []
    for tr, te in cv.split(np.zeros(len(y)), y):
        est = make_estimator().fit(X[tr], y[tr])
        rows.append(metrics(y[te], est.predict(X[te])))
    return pd.DataFrame(rows)


def run(encoder=None):
    """encoder: 문장 리스트 → (n, d) 배열 함수. None이면 SentenceTransformer를 로드."""
    if encoder is None:
        from sentence_transformers import SentenceTransformer
        t0 = time.time()
        st = SentenceTransformer(MODEL_NAME, device="cpu")
        print(f"모델 로드: {time.time() - t0:.1f}s")
        encoder = lambda texts: st.encode(texts, batch_size=64, normalize_embeddings=True,
                                          show_progress_bar=False)
    results, timing = [], []
    for size, path in FILES.items():
        df = pd.read_csv(path, encoding="utf-8-sig")
        texts, y = df["Text"].astype(str).values, df["Target"].values

        t0 = time.time(); emb = np.asarray(encoder(list(texts))); enc_sec = time.time() - t0
        timing.append({"Dataset": f"{size} (n={len(y)})", "Embedding dim": emb.shape[1],
                       "Encoding time (s)": round(enc_sec, 2),
                       "ms / sentence": round(1000 * enc_sec / len(y), 2)})

        for feat in ["TF-IDF", "SBERT MiniLM-L6"]:
            for m in ["LogReg", "LinearSVC"]:
                for bal in [False, True]:
                    if feat == "TF-IDF":
                        X = texts
                        make = lambda: Pipeline([("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=1,
                                                                           sublinear_tf=True)),
                                                 ("clf", make_clf(m, bal))])
                    else:
                        X = emb
                        make = lambda: make_clf(m, bal)
                    t0 = time.time(); folds = cv_eval(make, X, y); fit_sec = time.time() - t0
                    row = {"Dataset": f"{size} (n={len(y)})", "Features": feat, "Model": m,
                           "Imbalance": "balanced" if bal else "none"}
                    for c in folds.columns:
                        row[c] = f"{folds[c].mean():.3f} ± {folds[c].std():.3f}"
                    row["CV time (s)"] = round(fit_sec, 1)
                    results.append(row)

    res = pd.DataFrame(results)
    print("\n=== 인코딩 비용 ===")
    print(pd.DataFrame(timing).to_string(index=False))
    print(f"환경: {platform.platform()} | CPU 런타임 기준")
    print("\n=== 결과 (25개 폴드 평균 ± 표준편차) ===")
    print(res.to_string(index=False))
    res.to_csv("sbert_vs_tfidf_results.csv", index=False, encoding="utf-8-sig")
    print("\n저장: sbert_vs_tfidf_results.csv")
    return res


if __name__ == "__main__":
    run()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

모델 로드: 0.4s

=== 인코딩 비용 ===
      Dataset  Embedding dim  Encoding time (s)  ms / sentence
Small (n=125)            384               1.90          15.20
Large (n=480)            384               5.91          12.31
환경: Linux-6.6.159+-x86_64-with-glibc2.39 | CPU 런타임 기준

=== 결과 (25개 폴드 평균 ± 표준편차) ===
      Dataset        Features     Model Imbalance      Accuracy Bad Precision    Bad Recall        Bad F1      Macro-F1  CV time (s)
Small (n=125)          TF-IDF    LogReg      none 0.752 ± 0.016 0.000 ± 0.000 0.000 ± 0.000 0.000 ± 0.000 0.429 ± 0.005          0.5
Small (n=125)          TF-IDF    LogReg  balanced 0.813 ± 0.075 0.660 ± 0.206 0.567 ± 0.198 0.593 ± 0.169 0.735 ± 0.107          0.5
Small (n=125)          TF-IDF LinearSVC      none 0.814 ± 0.061 0.798 ± 0.283 0.346 ± 0.224 0.449 ± 0.224 0.668 ± 0.128          0.3
Small (n=125)          TF-IDF LinearSVC  balanced 0.818 ± 0.068 0.662 ± 0.187 0.542 ± 0.236 0.576 ± 0.191 0.729 ± 0.115          0.3
Small (n=125) SBERT MiniLM-L6    